# Synthetic Lethality Screening Strategy Comparison

This notebook evaluates how different SL screening parameters affect the number of significant hits. Configurations tested include:
- Biomarker clustering (on/off)
- Amplification thresholds
- WT definitions
- Effect size cutoffs

## Step 1A: Define Screening Configurations

In this block, we define the hyperparameters for each SL screening strategy.

Each config includes:
- `amp_threshold`: Minimum CNA value to define "Amplified"
- `wt_values`: CNA values considered as "WT"
- `cluster`: Whether to collapse biomarkers with identical AMP/WT profiles
- `name`: Used to name output folders and identify runs

In [35]:
# Required packages
import pandas as pd
import numpy as np

# Load input datasets:
## AMP biomarkers
amp_biomarkers = pd.read_csv("/Users/faith/Documents/GitHub/bioslate-hgsoc-core/results/cross_val_amp_sig_genes.csv", index_col=1)
amp_biomarkers = amp_biomarkers.index.astype(str)
amp_biomarkers

## CNA DepMap
cna_hgsoc_depmap = pd.read_csv("/Users/faith/Documents/GitHub/bioslate-hgsoc-core/results/cna_depmap_hgsoc.csv", index_col=0)
cna_hgsoc_depmap = cna_hgsoc_depmap.transpose()
cna_hgsoc_depmap_filtered = cna_hgsoc_depmap.loc[amp_biomarkers]
cna_hgsoc_depmap_filtered = cna_hgsoc_depmap_filtered.transpose()

## CRISPR
crispr_filtered = pd.read_csv("../streamlit_app/data/crispr_for_streamlit.csv", index_col=0) 

In [63]:
configs = []

wt_variants = [
    [2],
    [1, 2],
    [2, 3],
    [0, 1, 2],
    [0, 1, 2, 3],
    [0, 1, 2, 3, 4],
    [0, 1, 2, 3, 4, 5]
]

for wt in wt_variants:
    for cluster in [True]: # changed to True only for quickness
        label = "_".join(map(str, wt))
        cluster_tag = "Clustered" if cluster else "NoCluster"
        config = {
            "name": f"CNA6_WT{label}_{cluster_tag}",
            "amp_threshold": 6,
            "wt_values": wt,
            "cluster": cluster
        }
        configs.append(config)

## Step 2A: Define the Synthetic Lethality Screening Function (T-Test Function)

This function runs the full SL screening pipeline for a single configuration:
- Assigns CNA-based Amplified/WT status for each biomarker gene across cell lines
- Optionally collapses redundant biomarkers using CNA profile clustering
- Performs two-sample Welch’s t-tests comparing CRISPR gene effect scores between Amplified vs WT groups
- Calculates effect size (Cohen’s d) for each biomarker–target pair
- Applies multiple testing correction using Benjamini–Hochberg FDR
- Filters results based on significance (p-value < 0.05), FDR (< 0.1), effect size (< 0), and WT selectivity (> –1)
- Saves all result tables into a dedicated folder named after the config

The function returns a summary dictionary reporting the number of tests performed and the number of hits at various filtering stages.

In [64]:
import os
from collections import defaultdict
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

def run_sl_screen(config, amp_biomarkers, cna_df, crispr_df, out_root="../results"):
    amp_threshold = config["amp_threshold"]
    wt_values = config["wt_values"]
    cluster = config["cluster"]
    name = config["name"]

    out_dir = os.path.join(out_root, name)
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(os.path.join(out_dir, "other"), exist_ok=True)

    status_dict = {}
    for gene in amp_biomarkers:
        if gene in cna_df.columns:
            status_dict[gene] = cna_df[gene].apply(
                lambda x: "Amplified" if x >= amp_threshold else ("WT" if x in wt_values else np.nan)
            )
    status_df = pd.DataFrame(status_dict, index=cna_df.index)

    if cluster:
        profile_groups = defaultdict(list)
        for gene in status_df.columns:
            profile = tuple(status_df[gene].fillna("NA").values)
            profile_groups[profile].append(gene)
        biomarker_sets = [ (group[0], ", ".join(sorted(group))) for group in profile_groups.values() ]
    else:
        biomarker_sets = [ (gene, gene) for gene in status_df.columns ]

    results = []
    for rep_gene, alias in biomarker_sets:
        if rep_gene not in status_df:
            continue

        status = status_df[rep_gene]
        amp_lines = status[status == "Amplified"].index.intersection(crispr_df.index)
        wt_lines = status[status == "WT"].index.intersection(crispr_df.index)

        if len(amp_lines) < 3 or len(wt_lines) < 3:
            continue

        for target in crispr_df.columns:
            if not (crispr_df[target] < -0.6).any():
                continue

            group_amp = crispr_df.loc[amp_lines, target].dropna()
            group_wt = crispr_df.loc[wt_lines, target].dropna()
            if len(group_amp) < 3 or len(group_wt) < 3:
                continue

            t_stat, p_val = ttest_ind(group_amp, group_wt, equal_var=False)
            pooled_sd = np.sqrt(((group_amp.std(ddof=1)**2 + group_wt.std(ddof=1)**2) / 2))
            effect_size = (group_amp.mean() - group_wt.mean()) / pooled_sd if pooled_sd > 0 else np.nan

            results.append({
                "Biomarker": rep_gene,
                "BiomarkerCluster": alias,
                "TargetGene": target,
                "OncogeneAddiction": rep_gene == target,
                "T-stat": t_stat,
                "P-value": p_val,
                "EffectSize": effect_size,
                "n_Amplified": len(group_amp),
                "n_WT": len(group_wt),
                "MeanEffect_Amplified": group_amp.mean(),
                "MeanEffect_WT": group_wt.mean()
            })

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df["FDR"] = multipletests(results_df["P-value"], method="fdr_bh")[1]
    else:
        results_df["FDR"] = []

    results_df.to_csv(f"{out_dir}/synthetic_lethality_screen.csv", index=False)

    pval_hits = results_df[results_df["P-value"] < 0.05]
    pval_hits.to_csv(f"{out_dir}/other/significant_synthetic_hits_pval.csv", index=False)

    fdr_hits = results_df[results_df["FDR"] < 0.1]
    fdr_hits.to_csv(f"{out_dir}/other/significant_synthetic_hits_fdr.csv", index=False)

    strong_hits = fdr_hits[fdr_hits["EffectSize"] < 0]
    strong_hits.to_csv(f"{out_dir}/strong_synthetic_lethal_hits.csv", index=False)

    selective_hits = strong_hits[strong_hits["MeanEffect_WT"] > -1]
    selective_hits.to_csv(f"{out_dir}/selective_synthetic_lethal_hits.csv", index=False)

    return {
        "ConfigName": name,
        "SLTests": len(results_df),
        "FDRHits": len(fdr_hits),
        "StrongHits": len(strong_hits),
        "SelectiveHits": len(selective_hits)
    }


## Step 3: Run All Configurations and Collect Summary Stats

This loop:
- Runs `run_sl_screen()` for each config in the list
- Prints which config is currently running
- Stores the summary stats (e.g. number of SL tests, FDR hits, etc.)
- Outputs a `summary_df` DataFrame showing comparative results
- Saves the summary to a CSV for downstream reporting

Expect each config to take a few seconds to run depending on your biomarker and CRISPR data size.

In [65]:
summary = []

for cfg in configs:
    print(f"Running config: {cfg['name']}")
    result = run_sl_screen(cfg, amp_biomarkers, cna_hgsoc_depmap_filtered, crispr_filtered)
    summary.append(result)

summary_df = pd.DataFrame(summary)
summary_df.to_csv("../results/sl_screen_summary.csv", index=False)
summary_df

Running config: CNA6_WT2_Clustered
Running config: CNA6_WT1_2_Clustered
Running config: CNA6_WT2_3_Clustered
Running config: CNA6_WT0_1_2_Clustered
Running config: CNA6_WT0_1_2_3_Clustered
Running config: CNA6_WT0_1_2_3_4_Clustered


KeyboardInterrupt: 

## Step 4: Add HGNC Gene Symbols to SL Results

This step maps Entrez Gene IDs in the `Biomarker` and `TargetGene` columns to their official HGNC gene symbols using the `gene_with_protein_product.txt` reference file.

The HGNC names are inserted directly beside each ID column in both:
- `strong_synthetic_lethal_hits.csv`
- `selective_synthetic_lethal_hits.csv`

Updated files are saved with `_with_HGNC` suffix in the `../results/` folder.


In [59]:
import pandas as pd

# Load your result files
sl_hits_strong = pd.read_csv("../results/T-Test/CNA6_WTlt6_Clustered/strong_synthetic_lethal_hits.csv")
sl_hits_selective = pd.read_csv("../results/T-Test/CNA6_WTlt6_Clustered/selective_synthetic_lethal_hits.csv")

# Load the HGNC mapping file
hgnc_df = pd.read_csv("../database_files/gene_with_protein_product.txt", sep="\t")

# Build a mapping: Entrez ID (as string) → HGNC symbol
entrez_to_symbol = dict(zip(hgnc_df["entrez_id"].astype(str), hgnc_df["symbol"]))

# Function to insert HGNC symbols beside original columns
def insert_hgnc_columns(df):
    df.insert(
        loc=df.columns.get_loc("Biomarker") + 1,
        column="Biomarker_HGNC",
        value=df["Biomarker"].apply(lambda x: entrez_to_symbol.get(str(x)))
    )

    df.insert(
        loc=df.columns.get_loc("TargetGene") + 1,
        column="TargetGene_HGNC",
        value=df["TargetGene"].apply(lambda x: entrez_to_symbol.get(str(x)))
    )

    df.insert(
        loc=df.columns.get_loc("BiomarkerCluster") + 1,
        column="BiomarkerCluster_HGNC",
        value=df["BiomarkerCluster"].apply(
            lambda cluster: ", ".join([
                entrez_to_symbol.get(gid.strip(), "NA")
                for gid in cluster.split(",")
            ])
        )
    )
    return df

# Apply to both DataFrames
sl_hits_strong = insert_hgnc_columns(sl_hits_strong)
sl_hits_selective = insert_hgnc_columns(sl_hits_selective)

# Save updated files
sl_hits_strong.to_csv("../results/T-Test/CNA6_WTlt6_Clustered/strong_synthetic_lethal_hits_with_HGNC.csv", index=False)
sl_hits_selective.to_csv("../results/T-Test/CNA6_WTlt6_Clustered/selective_synthetic_lethal_hits_with_HGNC.csv", index=False)

/var/folders/09/0r9l07110lg5nj9ndgsx9kl80000gn/T/ipykernel_7466/920875569.py:8: DtypeWarning: Columns (35,38,45,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc_df = pd.read_csv("../database_files/gene_with_protein_product.txt", sep="\t")


## Step 1B: Define Screening Configurations

In this block, we define the hyperparameters for each SL screening strategy.

Each config includes:
- `amp_threshold`: Minimum CNA value to define "Amplified"
- `wt_values`: CNA values considered as "WT"
- `cluster`: Whether to collapse biomarkers with identical AMP/WT profiles
- `name`: Used to name output folders and identify runs

In [90]:
configs = [
    {"name": "CNA_quantitative"}
]


## Step 2B: Define the Synthetic Lethality Screening Function (Quantitative CNA Regression)

This function executes the SL screening pipeline using continuous CNA values as predictors in a linear regression framework:

* For each biomarker gene, raw copy number values (continuous) across cell lines are used directly as predictors.
* No discrete binning into 'Amplified' or 'WT' — the model assesses the quantitative relationship between CNA and CRISPR gene dependency.
* For every biomarker–target gene pair:

  * Fits an OLS linear regression: `CRISPR_dependency ~ CNA_value`
  * Computes the regression β coefficient, reflecting the change in gene effect score per unit increase in CNA.
  * Uses HC3 robust standard errors to account for heteroskedasticity (unequal variance across cell lines).
  * Extracts confidence intervals, p-values, and standardised effect size (β / SD of response).
  * Predicts dependency score at CNA = 2 to evaluate selectivity.

* Applies Benjamini–Hochberg FDR correction on p-values.

* Applies final hit filters:

  * **p < 0.05**
  * **FDR < 0.1**
  * **Negative effect size (β < 0)** — consistent with increased dependency in amplified contexts
  * **Predicted dependency at CNA = 2 > –1** — avoids pan-essential genes

* Saves all results to a dedicated output folder per configuration.

The function returns a dictionary summarising:

* Total gene pairs tested
* FDR-significant hits
* Strong hits (β < 0)
* Selective hits (non-pan-essential) for downstream prioritisation.


In [77]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

def run_sl_screen(config, amp_biomarkers, cna_df, crispr_df, out_root="../results"):
    name = config["name"]

    out_dir = os.path.join(out_root, name)
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(os.path.join(out_dir, "other"), exist_ok=True)

    # Use continuous CNA directly
    status_df = cna_df[amp_biomarkers.intersection(cna_df.columns)]
    biomarker_sets = [(gene, gene) for gene in status_df.columns]

    results = []
    for rep_gene in biomarker_sets:
        if rep_gene not in cna_df.columns:
            continue

        cna_vector = cna_df[rep_gene].dropna()
        common_lines = cna_vector.index.intersection(crispr_df.index)

        if len(common_lines) < 6:
            continue

        cna_vector = cna_vector.loc[common_lines]

        for target in crispr_df.columns:
            dep_vector = crispr_df[target].loc[common_lines].dropna()
            matched = dep_vector.index.intersection(cna_vector.index)

            if len(matched) < 6:
                continue

            # Require at least one strongly dependent cell line
            if not (dep_vector.loc[matched] < -0.6).any():
                continue

            X = sm.add_constant(cna_vector.loc[matched])
            y = dep_vector.loc[matched]

            try:
                model = sm.OLS(y, X).fit(cov_type="HC3")
                beta = model.params.iloc[1]
                p_val = model.pvalues.iloc[1]
                t_stat = model.tvalues.iloc[1]
                conf_int = model.conf_int().iloc[1]
                effect_size = beta / y.std(ddof=1) if y.std(ddof=1) > 0 else np.nan
                predicted_at_cna2 = model.predict([1, 2])[0]
            except:
                continue

            results.append({
                "Biomarker": rep_gene,
                "TargetGene": target,
                "OncogeneAddiction": rep_gene == target,
                "Beta": beta,
                "T-stat": t_stat,
                "P-value": p_val,
                "EffectSize": effect_size,
                "CI_Lower": conf_int[0],
                "CI_Upper": conf_int[1],
                "n_Lines": len(matched),
                "MeanCNA": cna_vector.loc[matched].mean(),
                "MeanEffect": y.mean(),
                "PredictedEffect_CNA2": predicted_at_cna2
            })

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df["FDR"] = multipletests(results_df["P-value"], method="fdr_bh")[1]
    else:
        results_df["FDR"] = []

    results_df.to_csv(f"{out_dir}/synthetic_lethality_screen.csv", index=False)

    pval_hits = results_df[results_df["P-value"] < 0.05]
    pval_hits.to_csv(f"{out_dir}/other/significant_synthetic_hits_pval.csv", index=False)

    fdr_hits = results_df[results_df["FDR"] < 0.1]
    fdr_hits.to_csv(f"{out_dir}/other/significant_synthetic_hits_fdr.csv", index=False)

    strong_hits = fdr_hits[fdr_hits["EffectSize"] < 0]
    strong_hits.to_csv(f"{out_dir}/strong_synthetic_lethal_hits.csv", index=False)

    selective_hits = strong_hits[strong_hits["PredictedEffect_CNA2"] > -1]
    selective_hits.to_csv(f"{out_dir}/selective_synthetic_lethal_hits.csv", index=False)

    return {
        "ConfigName": name,
        "SLTests": len(results_df),
        "FDRHits": len(fdr_hits),
        "StrongHits": len(strong_hits),
        "SelectiveHits": len(selective_hits)
    }


## Step 2C: Handling Potential Co-Amplification Confounding

To avoid false positives from genes that are co-amplified together across cell lines:

* **Exact CNA profile clustering** is used:

  * Genes with *identical* CNA values across all profiled cell lines are grouped into a cluster.
  * Only one gene (the first) is retained as the representative biomarker per cluster.
  * This avoids redundancy when multiple genes share indistinguishable CNA patterns.

* This is a strict filter — it **does not** capture biologically co-amplified genes unless their profiles are identical.

* For broader co-amplification handling:

  * Consider computing **pairwise CNA correlations** across biomarkers.
  * Filter or annotate hits where biomarker–target pairs are frequently **co-amplified** in the same cell lines.

This step helps reduce artificial hits due to shared amplification patterns rather than genuine synthetic lethality.


In [98]:
configs = [
    {"name": "CNA_quantitative_cluster_delta"}
]

In [86]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from collections import defaultdict

def run_sl_screen(config, amp_biomarkers, cna_df, crispr_df, out_root="../results"):
    name = config["name"]

    out_dir = os.path.join(out_root, name)
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(os.path.join(out_dir, "other"), exist_ok=True)

    # Use continuous CNA directly
    status_df = cna_df[amp_biomarkers.intersection(cna_df.columns)]

    # Cluster biomarkers with identical CNA profiles
    profile_groups = defaultdict(list)
    for gene in status_df.columns:
        profile = tuple(status_df[gene].fillna("NA").values)
        profile_groups[profile].append(gene)
    biomarker_sets = [(group[0], ", ".join(sorted(group))) for group in profile_groups.values()]

    results = []
    for rep_gene, cluster_str in biomarker_sets:
        if rep_gene not in cna_df.columns:
            continue

        cna_vector = cna_df[rep_gene].dropna()
        common_lines = cna_vector.index.intersection(crispr_df.index)

        if len(common_lines) < 6:
            continue

        cna_vector = cna_vector.loc[common_lines]

        for target in crispr_df.columns:
            dep_vector = crispr_df[target].loc[common_lines].dropna()
            matched = dep_vector.index.intersection(cna_vector.index)

            if len(matched) < 6:
                continue

            # Require at least one strongly dependent cell line
            if not (dep_vector.loc[matched] < -0.6).any():
                continue

            X = sm.add_constant(cna_vector.loc[matched])
            y = dep_vector.loc[matched]

            try:
                model = sm.OLS(y, X).fit(cov_type="HC3")
                beta = model.params.iloc[1]
                p_val = model.pvalues.iloc[1]
                t_stat = model.tvalues.iloc[1]
                conf_int = model.conf_int().iloc[1]
                effect_size = beta / y.std(ddof=1) if y.std(ddof=1) > 0 else np.nan
                predicted_at_cna2 = model.predict([1, 2])[0]
            except:
                continue

            results.append({
                "Biomarker": rep_gene,
                "BiomarkerCluster": cluster_str,
                "TargetGene": target,
                "OncogeneAddiction": rep_gene == target,
                "Beta": beta,
                "T-stat": t_stat,
                "P-value": p_val,
                "EffectSize": effect_size,
                "CI_Lower": conf_int[0],
                "CI_Upper": conf_int[1],
                "n_Lines": len(matched),
                "MeanCNA": cna_vector.loc[matched].mean(),
                "MeanEffect": y.mean(),
                "PredictedEffect_CNA2": predicted_at_cna2
            })

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df["FDR"] = multipletests(results_df["P-value"], method="fdr_bh")[1]
    else:
        results_df["FDR"] = []

    results_df.to_csv(f"{out_dir}/synthetic_lethality_screen.csv", index=False)

    pval_hits = results_df[results_df["P-value"] < 0.05]
    pval_hits.to_csv(f"{out_dir}/other/significant_synthetic_hits_pval.csv", index=False)

    fdr_hits = results_df[results_df["FDR"] < 0.05]
    fdr_hits.to_csv(f"{out_dir}/other/significant_synthetic_hits_fdr.csv", index=False)

    strong_hits = fdr_hits[fdr_hits["EffectSize"] < 0]
    strong_hits.to_csv(f"{out_dir}/strong_synthetic_lethal_hits.csv", index=False)

    selective_hits = strong_hits[strong_hits["PredictedEffect_CNA2"] > -1]
    selective_hits.to_csv(f"{out_dir}/selective_synthetic_lethal_hits.csv", index=False)

    return {
        "ConfigName": name,
        "SLTests": len(results_df),
        "FDRHits": len(fdr_hits),
        "StrongHits": len(strong_hits),
        "SelectiveHits": len(selective_hits)
    }


In [99]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from collections import defaultdict

def run_sl_screen(config, amp_biomarkers, cna_df, crispr_df, out_root="../results"):
    name = config["name"]

    out_dir = os.path.join(out_root, name)
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(os.path.join(out_dir, "other"), exist_ok=True)

    # Use continuous CNA directly
    status_df = cna_df[amp_biomarkers.intersection(cna_df.columns)]

    # Cluster biomarkers with identical CNA profiles
    profile_groups = defaultdict(list)
    for gene in status_df.columns:
        profile = tuple(status_df[gene].fillna("NA").values)
        profile_groups[profile].append(gene)
    biomarker_sets = [(group[0], ", ".join(sorted(group))) for group in profile_groups.values()]

    results = []
    for rep_gene, cluster_str in biomarker_sets:
        if rep_gene not in cna_df.columns:
            continue

        cna_vector = cna_df[rep_gene].dropna()
        common_lines = cna_vector.index.intersection(crispr_df.index)

        if len(common_lines) < 6:
            continue

        cna_vector = cna_vector.loc[common_lines]

        for target in crispr_df.columns:
            dep_vector = crispr_df[target].loc[common_lines].dropna()
            matched = dep_vector.index.intersection(cna_vector.index)

            if len(matched) < 6:
                continue

            # Require at least one strongly dependent cell line
            if not (dep_vector.loc[matched] < -0.6).any():
                continue

            X = sm.add_constant(cna_vector.loc[matched])
            y = dep_vector.loc[matched]

            try:
                model = sm.OLS(y, X).fit(cov_type="HC3")
                beta = model.params.iloc[1]
                p_val = model.pvalues.iloc[1]
                t_stat = model.tvalues.iloc[1]
                conf_int = model.conf_int().iloc[1]
                effect_size = beta / y.std(ddof=1) if y.std(ddof=1) > 0 else np.nan
                predicted_at_cna2 = model.predict([1, 2])[0]

                # Mean effect in samples where CNA ≥ 6
                high_amp_lines = cna_vector[cna_vector >= 6].index
                mean_effect_amp6plus = y.loc[high_amp_lines].mean() if not y.loc[high_amp_lines].empty else np.nan

                delta = mean_effect_amp6plus - predicted_at_cna2 if pd.notnull(mean_effect_amp6plus) else np.nan
            except:
                continue

            results.append({
                "Biomarker": rep_gene,
                "BiomarkerCluster": cluster_str,
                "TargetGene": target,
                "OncogeneAddiction": rep_gene == target,
                "Beta": beta,
                "T-stat": t_stat,
                "P-value": p_val,
                "EffectSize": effect_size,
                "CI_Lower": conf_int[0],
                "CI_Upper": conf_int[1],
                "n_Lines": len(matched),
                "MeanCNA": cna_vector.loc[matched].mean(),
                "MeanEffect": y.mean(),
                "MeanEffect_CNA6plus": mean_effect_amp6plus,
                "PredictedEffect_CNA2": predicted_at_cna2,
                "DeltaEffect_CNA6minusPred2": delta
            })

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df["FDR"] = multipletests(results_df["P-value"], method="fdr_bh")[1]
    else:
        results_df["FDR"] = []

    results_df.to_csv(f"{out_dir}/synthetic_lethality_screen.csv", index=False)

    pval_hits = results_df[results_df["P-value"] < 0.05]
    pval_hits.to_csv(f"{out_dir}/other/significant_synthetic_hits_pval.csv", index=False)

    fdr_hits = results_df[results_df["FDR"] < 0.05]
    fdr_hits.to_csv(f"{out_dir}/other/significant_synthetic_hits_fdr.csv", index=False)

    strong_hits = fdr_hits[fdr_hits["EffectSize"] < 0]
    strong_hits.to_csv(f"{out_dir}/strong_synthetic_lethal_hits.csv", index=False)

    selective_hits = strong_hits[strong_hits["PredictedEffect_CNA2"] > -1]
    selective_hits.to_csv(f"{out_dir}/selective_synthetic_lethal_hits.csv", index=False)

    return {
        "ConfigName": name,
        "SLTests": len(results_df),
        "FDRHits": len(fdr_hits),
        "StrongHits": len(strong_hits),
        "SelectiveHits": len(selective_hits)
    }

## Step 3: Run All Configurations and Collect Summary Stats

This loop:
- Runs `run_sl_screen()` for each config in the list
- Prints which config is currently running
- Stores the summary stats (e.g. number of SL tests, FDR hits, etc.)
- Outputs a `summary_df` DataFrame showing comparative results
- Saves the summary to a CSV for downstream reporting

Expect each config to take a few seconds to run depending on your biomarker and CRISPR data size.

In [100]:
summary = []

for cfg in configs:
    print(f"Running config: {cfg['name']}")
    result = run_sl_screen(cfg, amp_biomarkers, cna_hgsoc_depmap_filtered, crispr_filtered)
    summary.append(result)

summary_df = pd.DataFrame(summary)
summary_df.to_csv("../results/sl_screen_summary.csv", index=False)
summary_df

Running config: CNA_quantitative_cluster_delta


,ConfigName,SLTests,FDRHits,StrongHits,SelectiveHits
0,CNA_quantitative_cluster_delta,521374,3476,1601,1075


## Filter to Most Potent Hits

In [14]:
import pandas as pd

selective_hits = pd.read_csv("../results/CNA_quantitative_cluster_delta/selective_synthetic_lethal_hits.csv")

# Define the exact desired column order
desired_order = [
    'Biomarker', 'BiomarkerCluster', 'TargetGene', 'OncogeneAddiction',
    'Beta', 'T-stat', 'P-value', 'EffectSize', 'CI_Lower', 'CI_Upper',
    'n_Lines', 'MeanCNA', 'MeanEffect',
    'MeanEffect_CNA6plus',  # <-- moved here
    'PredictedEffect_CNA2',  # <-- comes after
    'DeltaEffect_CNA6minusPred2',
    'FDR'
]

# Reorder
selective_hits = selective_hits[desired_order]
selective_hits

,Biomarker,BiomarkerCluster,TargetGene,OncogeneAddiction,Beta,T-stat,P-value,EffectSize,CI_Lower,CI_Upper,n_Lines,MeanCNA,MeanEffect,MeanEffect_CNA6plus,PredictedEffect_CNA2,DeltaEffect_CNA6minusPred2,FDR
0,10016,"10016, 11336",10480,False,-0.118552,-3.867643,1.098923e-04,-0.499304,-0.178629,-0.058475,17,4.058824,-1.154748,-1.517690,-0.910671,-0.607019,2.253932e-02
1,1004,"1004, 55322",7849,False,-0.197322,-4.376055,1.208464e-05,-0.301321,-0.285699,-0.108945,17,3.941176,-0.915860,-1.356943,-0.532824,-0.824120,4.393152e-03
2,1004,"1004, 55322",8467,False,-0.097358,-6.090735,1.123935e-09,-0.474773,-0.128687,-0.066029,17,3.941176,-0.458666,-0.705696,-0.269677,-0.436018,1.878175e-06
3,1004,"1004, 55322",7328,False,-0.115045,-3.915591,9.018315e-05,-0.358058,-0.172631,-0.057459,17,3.941176,-0.482106,-0.864307,-0.258783,-0.605524,1.945707e-02
4,10051,"10051, 3840, 51068, 8706",132660,False,-0.102927,-4.284172,1.834207e-05,-0.379547,-0.150015,-0.055839,17,4.588235,-0.290486,-0.573911,-0.024086,-0.549825,5.870519e-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1070,9846,9846,9804,False,-0.042765,-5.147339,2.642076e-07,-0.174569,-0.059049,-0.026481,17,5.097063,-0.616260,-0.934759,-0.483814,-0.450945,2.037736e-04
1071,9846,9846,163131,False,-0.057176,-7.890749,3.003784e-15,-0.261418,-0.071378,-0.042974,17,5.097063,-0.126525,-0.361195,0.050554,-0.411748,1.614531e-11
1072,9847,9847,55140,False,-0.117945,-3.911726,9.163899e-05,-0.358267,-0.177041,-0.058849,17,4.000000,-0.839405,-1.161180,-0.603516,-0.557664,1.967800e-02
1073,9847,9847,132660,False,-0.103511,-4.398658,1.089224e-05,-0.381700,-0.149633,-0.057388,17,4.000000,-0.290486,-0.573911,-0.083464,-0.490447,4.085562e-03


In [17]:
potent_hits = selective_hits[
    (selective_hits["DeltaEffect_CNA6minusPred2"] <= -0.5) &
    (selective_hits["PredictedEffect_CNA2"] >= -0.6)
]

potent_hits.to_csv("../results/CNA_quantitative_cluster_delta/potent_synthetic_lethal_hits.csv", index=False)

## Step 4: Add HGNC Gene Symbols to SL Results

This step maps Entrez Gene IDs in the `Biomarker` and `TargetGene` columns to their official HGNC gene symbols using the `gene_with_protein_product.txt` reference file.

The HGNC names are inserted directly beside each ID column in both:
- `strong_synthetic_lethal_hits.csv`
- `selective_synthetic_lethal_hits.csv`

Updated files are saved with `_with_HGNC` suffix in the `../results/` folder.


In [18]:
import pandas as pd

# Load your result files
sl_hits_potent = pd.read_csv("../results/CNA_quantitative_cluster_delta/potent_synthetic_lethal_hits.csv")

# Load the HGNC mapping file
hgnc_df = pd.read_csv("../database_files/gene_with_protein_product.txt", sep="\t")

# Build a mapping: Entrez ID (as string) → HGNC symbol
entrez_to_symbol = dict(zip(hgnc_df["entrez_id"].astype(str), hgnc_df["symbol"]))

# Function to insert HGNC symbols beside original columns
def insert_hgnc_columns(df):
    df.insert(
        loc=df.columns.get_loc("Biomarker") + 1,
        column="Biomarker_HGNC",
        value=df["Biomarker"].apply(lambda x: entrez_to_symbol.get(str(x)))
    )

    df.insert(
        loc=df.columns.get_loc("TargetGene") + 1,
        column="TargetGene_HGNC",
        value=df["TargetGene"].apply(lambda x: entrez_to_symbol.get(str(x)))
    )

    df.insert(
        loc=df.columns.get_loc("BiomarkerCluster") + 1,
        column="BiomarkerCluster_HGNC",
        value=df["BiomarkerCluster"].apply(
            lambda cluster: ", ".join([
                entrez_to_symbol.get(gid.strip(), "NA")
                for gid in cluster.split(",")
            ])
        )
    )
    return df

# Apply to DataFrame
sl_hits_potent = insert_hgnc_columns(sl_hits_potent)

# Save updated files
sl_hits_potent.to_csv("../results/CNA_quantitative_cluster_delta/potent_synthetic_lethal_hits_with_HGNC.csv", index=False)

/var/folders/09/0r9l07110lg5nj9ndgsx9kl80000gn/T/ipykernel_43025/1777138762.py:7: DtypeWarning: Columns (35,38,45,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc_df = pd.read_csv("../database_files/gene_with_protein_product.txt", sep="\t")


## Clean Hits for Agent

In [20]:
import pandas as pd

# Load your full potent hits file
input_file = "../results/CNA_quantitative_cluster_delta/potent_synthetic_lethal_hits_with_HGNC.csv"
df = pd.read_csv(input_file)

# Extract required columns for the agent
agent_input_df = df[["BiomarkerCluster_HGNC", "TargetGene_HGNC"]].copy()

# Save for agent use
output_file = "../agents/assets/cluster_hits.csv"
agent_input_df.to_csv(output_file, index=False)

print(f"Cleaned agent input saved to: {output_file}")

Cleaned agent input saved to: ../agents/assets/cluster_hits.csv
